In [2]:
!pip install confluent-kafka faker python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.6 MB/s eta 0:00:00


In [ ]:
import os

os.environ['BOOTSTRAP_SERVERS'] = 'Kafka Server'
os.environ['API_KEY'] = 'Kafka API Key'
os.environ['API_SECRET'] = 'Kafka Secret Key'

In [4]:
from confluent_kafka import Producer
import os

conf = {
    'bootstrap.servers': os.getenv('BOOTSTRAP_SERVERS'),
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': os.getenv('API_KEY'),
    'sasl.password': os.getenv('API_SECRET')
}

producer = Producer(conf)

def delivery_report(err, msg):
    if err:
        print(f"❌ Delivery failed: {err}")
    else:
        print(f"✅ Message sent to {msg.topic()}")

producer.produce("smartgear_orders", key="test", value="Hello from Colab!", callback=delivery_report)
producer.flush()

✅ Message sent to smartgear_orders


0

## 📊 SmartGear Streaming Data Generator – Design & Implementation

### 🎯 Objective

The goal of this component is to simulate a **real-time streaming data source** for SmartGear retail orders, which will be ingested into Kafka and processed using a Lakehouse architecture.

---

## 🧠 Design Philosophy

Instead of generating completely random data, we follow a **data-driven simulation approach**:

* The streaming data **mimics the statistical distribution** of the historical batch dataset (`smartgear_sales.csv`)
* This ensures:

  * Realistic data generation
  * Meaningful analytics in downstream layers
  * Ability to perform **batch vs streaming reconciliation**

---

## 📌 Key Design Decisions

### 1. Product-Based Pricing Distribution

Each product has a predefined **price range derived from batch data analysis**:

| Product        | Price Range |
| -------------- | ----------- |
| Camera         | 450 – 550   |
| Drone          | 630 – 770   |
| Gaming Console | 400 – 500   |
| Headphones     | 70 – 90     |
| Laptop         | 720 – 880   |
| Monitor        | 180 – 220   |
| Printer        | 135 – 165   |
| Smartphone     | 540 – 660   |
| Smartwatch     | 225 – 275   |
| Tablet         | 360 – 440   |

👉 This ensures:

* Streaming data aligns with real-world patterns
* Aggregations like revenue, averages, and trends are meaningful

---

### 2. Event Schema

Each Kafka message follows this schema:

```json
{
  "order_id": "UUID",
  "timestamp": "ISO Timestamp",
  "store_id": "store_X",
  "product": "Product Name",
  "quantity": Integer,
  "price": Float,
  "region": "north/south/east/west"
}
```

---

### 3. Real-Time Simulation Strategy

* Events are generated continuously
* Data is sent in **micro-batches (5–10 records)**
* A delay of **5 seconds** simulates real-world streaming

---

### 4. Partitioning Strategy

* Kafka key = `order_id`
* Ensures:

  * Even distribution across partitions
  * Event uniqueness
  * Scalability

---

### 5. Why Not Random Data?

Pure random generation would:

* Break consistency with batch data
* Produce unrealistic KPIs
* Make reconciliation impossible

👉 Therefore, we use **controlled randomness within realistic bounds**

---

## 🚀 Implementation Code


In [7]:
from confluent_kafka import Producer
from faker import Faker
import os, json, time, random
from datetime import datetime

fake = Faker()

# Kafka Configuration
conf = {
    'bootstrap.servers': os.getenv('BOOTSTRAP_SERVERS'),
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': os.getenv('API_KEY'),
    'sasl.password': os.getenv('API_SECRET')
}

producer = Producer(conf)

# Product Price Mapping (Derived from Batch Data)
product_price_map = {
    "Camera": (450, 550),
    "Drone": (630, 770),
    "Gaming Console": (400, 500),
    "Headphones": (70, 90),
    "Laptop": (720, 880),
    "Monitor": (180, 220),
    "Printer": (135, 165),
    "Smartphone": (540, 660),
    "Smartwatch": (225, 275),
    "Tablet": (360, 440)
}

products = list(product_price_map.keys())
regions = ["North", "South", "East", "West"]
stores = [f"{i}" for i in range(101, 121)]  # matching your pivot

order_counter = 2001
# Generate Order
def generate_order():
    product = random.choice(products)
    price_range = product_price_map[product]

    return {
        "order_id": str(fake.uuid4()),
        "order_number": order_counter,
        "timestamp": datetime.utcnow().isoformat(),
        "store_id": random.choice(stores),
        "product": product,
        "quantity": random.randint(1, 5),
        "price": round(random.uniform(*price_range), 2),
        "region": random.choice(regions)
    }

# Delivery Callback
def delivery_report(err, msg):
    if err:
        print(f"❌ Failed: {err}")
    else:
        print(f"✅ Sent: {msg.key()}")

# Streaming Loop
while True:
    batch_size = random.randint(5, 10)

    for _ in range(batch_size):
        order = generate_order()

        producer.produce(
            topic="smartgear_orders",
            key=order["order_id"],
            value=json.dumps(order),
            callback=delivery_report
        )

    producer.flush()

    print(f"🚀 Batch of {batch_size} events sent")
    time.sleep(5)

/tmp/ipykernel_16395/2230875809.py:46: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


✅ Sent: b'9c98c337-3550-430c-82f8-3721a24fa967'
✅ Sent: b'324fd3cc-a382-43c1-afd8-5d0b64ebfc7f'
✅ Sent: b'4d844e65-2ded-4f17-9caf-0017d924d978'
✅ Sent: b'd2af783d-7308-4cf2-9516-4be0c1b75103'
✅ Sent: b'8f90b833-1f05-47f4-8935-5b383fc7416b'
🚀 Batch of 5 events sent
✅ Sent: b'6147be10-29cc-4af0-b3ca-816de90247d6'
✅ Sent: b'69170530-fcff-4dd7-9280-7940e26b6f74'
✅ Sent: b'4a6c1433-8d7b-4d84-a3d1-82eac01ccd78'
✅ Sent: b'9445fe49-6a61-4343-b585-a07d762786b9'
✅ Sent: b'855cd7f9-080d-48e4-aef3-41f15fc2179d'
✅ Sent: b'5b765a3e-6aaf-4f02-848c-04b6ed2ef6f8'
✅ Sent: b'85b27674-1fe4-4c61-b34a-62886e84a3de'
✅ Sent: b'fa9c8eab-e6ca-45e4-83e0-cb36aa8d5a3e'
✅ Sent: b'bee09f51-1db8-464f-837b-e85035d8a8f5'
✅ Sent: b'86d22777-1e31-4456-99d5-9a57955b2f27'
🚀 Batch of 10 events sent


KeyboardInterrupt: 